# Beamforming Evaluation — Fixed Twin, Array Sweep, Lead Time

Carries the predictors from `5_minlstm_study.ipynb` into the beamforming metric,
after repairing the two defects that made the original SNR numbers
uninformative.

### Defect 1 — the digital-twin lookup returned the true future channel

`3_beamforming.ipynb` resolved a predicted position against
`positions_2d[:current_t+2]`:

```python
dists = np.sqrt(np.sum((positions_2d[:current_t+2, :2] - target_pos)**2, axis=1))
nearest_idx = np.argmin(dists)
```

That slice **includes index `t+1`**, the true next position. Trajectory points
sit ~1.5 m apart while prediction error is ~0.1 m, so `argmin` returned `t+1`
essentially every time and the "predicted" channel *was* the true future
channel. SNR became a step function of prediction error with a threshold at
half the step size — which is why MAE 0.104 and 0.154 produced identical gains,
and why only the zigzag case (MAE 0.703, straddling the threshold) ever lost.

§2 measures how often the old lookup cheated before replacing it.

### Defect 2 — the array was too small to resolve anything

An 8-element ULA has a ~12.75° half-power beamwidth. A 1.6 m step at 200 m
range subtends 0.46°, i.e. **3.6% of a beamwidth** — no predictor difference can
register. At Nt=128 (0.80° HPBW) the same step is **57% of a beamwidth**.

The predictor is position-only, so it never sees an antenna. The array sweep
therefore costs nothing at training time: predict once, evaluate at every Nt.

## 1. Setup

In [ ]:
import sys, pathlib, json, time
import numpy as np, torch, matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.train import resolve_device, predict, set_seed
from src.models import build_model
from src.twin import DigitalTwin
from src.beamforming import (mrt_weights, snr, snr_db, spectral_efficiency,
                             calibrate_noise, half_power_beamwidth_deg,
                             misalignment_deg, misalignment_beamwidths)
from src.trajectories import generate_corpus
from src.data import build_dataset, split_by_trajectory
from src.baselines import kalman_cv, ls_linear

DEVICE = resolve_device()
RESULTS = ROOT / "results"; MODELS = ROOT / "models"
STUDY = json.loads((RESULTS / "minlstm_study.json").read_text())
CFG = STUDY["config"]
print(f"device {DEVICE} | study config: {CFG}")

BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9c8c2", "axes.grid": True,
    "grid.color": "#dedcd5", "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 10,
    "lines.linewidth": 2.0, "legend.frameon": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
})

In [ ]:
# Same seed as notebook 5 -> identical corpus, no need to serialise it.
rng = np.random.default_rng(CFG["seed"])
corpus = generate_corpus(n=CFG["n_traj"], steps=CFG["steps"], rng=rng,
                         sigma=CFG["sigma"], dt=CFG["dt"])
train_c, val_c, test_c = split_by_trajectory(corpus, (0.7, 0.15, 0.15), rng=rng)
print(f"{len(test_c)} test trajectories (regenerated from seed {CFG['seed']})")

## 2. The leak, measured

Before replacing the old lookup, quantify it: across the test trajectories, how
often did resolving against `positions[:t+2]` return exactly index `t+1`? If
that rate is near 100%, the published SNR gains were measuring an oracle.

In [ ]:
W = STUDY["models"]["lstm"]["params"]["window"]

def legacy_lookup_index(traj_positions, target, t):
    """The original behaviour, reproduced exactly for measurement."""
    d = np.linalg.norm(traj_positions[:t + 2, :2] - target[:2], axis=1)
    return int(np.argmin(d))

# Use the tuned Kalman as a stand-in predictor with realistic error.
kf = kalman_cv(dt=CFG["dt"], sigma=CFG["sigma"], q=STUDY["classical"]["q"])
hits, total = 0, 0
for traj in test_c:
    pos, obs = traj["true"], traj["observed"]
    for t in range(W - 1, len(pos) - 2):
        pred = kf(obs[t - W + 1:t + 1])
        hits += legacy_lookup_index(pos, pred, t) == t + 1
        total += 1

print(f"legacy lookup returned the TRUE future index in {hits/total:.1%} of steps "
      f"({hits:,}/{total:,})")
print("-> the 'predictive' branch was reading the answer, not predicting it")

## 3. Digital twin over the full grid

Loading `O1_60` and indexing the real 497,931-point receiver grid. Channels are
computed lazily and memoised — a whole study touches only a few thousand points,
so precomputing the grid would waste hours.

In [ ]:
import deepmimo as dm

dataset = dm.load("o1_60")
bs0 = dataset[0]
grid = np.asarray(bs0["rx_pos"])
print(f"grid: {grid.shape[0]:,} points | "
      f"x {grid[:,0].min():.1f}-{grid[:,0].max():.1f} m | "
      f"y {grid[:,1].min():.1f}-{grid[:,1].max():.1f} m")

In [ ]:
# The base-station position. The original notebook assumed the origin, which is
# wrong for this coordinate frame and corrupts every angle it reports.
for key in ("tx_pos", "bs_pos", "txrx_pos"):
    if key in getattr(bs0, "keys", lambda: [])():
        BS_POS = np.asarray(bs0[key]).reshape(-1)[:2]
        print(f"base station from bs0['{key}']: {BS_POS}")
        break
else:
    raise KeyError(
        f"Could not find the TX position. Available keys: {sorted(bs0.keys())}\n"
        "Set BS_POS manually from the scenario description before continuing — "
        "every angle below depends on it."
    )

### Array size

DeepMIMO exposes the TX array geometry through the channel parameters. The cell
below prints what the installed version actually offers rather than assuming an
API: confirm the antenna-count field before running the sweep, because a silent
fallback to the default 8 elements would make §5 look like a null result.

In [ ]:
params = bs0.ch_params if hasattr(bs0, "ch_params") else None
print("channel parameter object:", type(params).__name__ if params else "not found")
if params is not None:
    for k in dir(params):
        if not k.startswith("_") and any(s in k.lower() for s in ("ant", "array", "bs", "tx")):
            print(f"  {k} = {getattr(params, k, '?')}")

In [ ]:
def channels_for(indices, n_tx):
    """Compute channels for a batch of grid indices at a given TX array size.

    Adjust the two marked lines to match the printed parameter names above if
    your DeepMIMO build differs.
    """
    sub = bs0.trim(idxs=list(indices))
    p = sub.ch_params
    p.bs_antenna.shape = np.array([n_tx, 1])   # <-- confirm against the print above
    out = np.asarray(sub.compute_channels(p))
    assert out.shape[2] == n_tx, (
        f"asked for {n_tx} TX antennas, got {out.shape[2]} — the parameter name "
        "above is wrong and the sweep would be meaningless"
    )
    return out

## 4. Predictors

Four comparators plus two reference points. The references matter as much as the
models - without them an SNR number has no scale:

- **reactive** - beam at the last *observed* position. The floor.
- **oracle** - beam computed from the true channel at `t+1`. The ceiling no
  predictor can pass, and the thing the old buggy lookup was accidentally
  reporting.

In [ ]:
def load_predictor(arch):
    p = STUDY["models"][arch]["params"]
    model = build_model(arch, hidden_size=p["hidden"], num_layers=p["layers"],
                        dropout=p["dropout"])
    model.load_state_dict(torch.load(MODELS / f"{arch}_best.pt", map_location="cpu"))
    return model.to(DEVICE).eval(), p["window"]


def model_predictions(arch, trajs):
    """Absolute predicted positions, aligned to targets t = W .. T-1."""
    model, W = load_predictor(arch)
    _, _, _, sc = build_dataset(train_c, W)          # scales from training data only
    X, _, anchors, _ = build_dataset(trajs, W, scales=sc)
    rel = predict(model, X, device=DEVICE) * sc["y"]
    return anchors + rel, W


def baseline_predictions(fn, trajs, W):
    return np.array([fn(t["observed"][i:i+W])
                     for t in trajs for i in range(len(t["observed"]) - W)]), W


PREDICTORS = {}
for arch in ("lstm", "minlstm"):
    try:
        PREDICTORS[arch] = model_predictions(arch, test_c)
    except FileNotFoundError:
        print(f"skipping {arch}: run notebook 5 first")

W_REF = STUDY["classical"]["W"]
PREDICTORS["kalman"] = baseline_predictions(
    kalman_cv(dt=CFG["dt"], sigma=CFG["sigma"], q=STUDY["classical"]["q"]), test_c, W_REF)
PREDICTORS["ls_linear"] = baseline_predictions(ls_linear, test_c, W_REF)
PREDICTORS["reactive"] = (np.concatenate(
    [t["observed"][W_REF-1:len(t["observed"])-1] for t in test_c]), W_REF)

for name, (pred, W) in PREDICTORS.items():
    print(f"{name:<12} {len(pred):>7,} predictions (W={W})")

In [ ]:
def truths_for(W):
    return np.concatenate([t["true"][W:] for t in test_c])

print(f"{'predictor':<12}{'MAE (m)':>10}{'misalign (deg)':>17}")
for name, (pred, W) in PREDICTORS.items():
    truth = truths_for(W)
    mae = np.mean(np.linalg.norm(pred - truth, axis=1))
    ang = np.mean([misalignment_deg(p, t, BS_POS) for p, t in zip(pred, truth)])
    print(f"{name:<12}{mae:>10.3f}{ang:>17.4f}")

## 5. Array-size sweep

The same predictions evaluated at Nt = 8 ... 128. Beam misalignment is reported
in **half-power beamwidths** rather than degrees, because degrees are not
comparable across array sizes and beamwidths are exactly what determines
whether the user is still inside the beam.

In [ ]:
N_TX = [8, 16, 32, 64, 128]

# One channel computation per (grid point, Nt): gather every index first.
index_of = DigitalTwin(grid, lambda i: None)
needed = set()
for name, (pred, W) in PREDICTORS.items():
    needed |= set(index_of.nearest_indices(pred).tolist())
    needed |= set(index_of.nearest_indices(truths_for(W)).tolist())
needed = sorted(needed)
print(f"{len(needed):,} unique grid points touched "
      f"({len(needed)/len(grid):.2%} of the scenario)")

In [ ]:
twins, noise = {}, {}
for n_tx in N_TX:
    t0 = time.time()
    twin = DigitalTwin(grid, lambda idx, n=n_tx: channels_for(idx, n))
    twin.prefetch(needed)
    twins[n_tx] = twin
    noise[n_tx] = calibrate_noise([twin.channel_for_index(i) for i in needed[:2000]],
                                  target_snr_db=20.0)
    print(f"Nt={n_tx:<4} cached {twin.n_cached:,} channels in {time.time()-t0:.0f}s "
          f"| HPBW {half_power_beamwidth_deg(n_tx):.2f} deg")

In [ ]:
rows = []
for n_tx in N_TX:
    twin = twins[n_tx]
    for name, (pred, W) in PREDICTORS.items():
        truth = truths_for(W)
        true_idx = twin.nearest_indices(truth)
        pred_idx = twin.nearest_indices(pred)

        snrs, mis = [], []
        for pi, ti, p, t in zip(pred_idx, true_idx, pred, truth):
            w = mrt_weights(twin.channel_for_index(pi))                  # PREDICTED channel
            snrs.append(snr(twin.channel_for_index(ti), w, noise[n_tx]))  # on TRUE channel
            mis.append(misalignment_beamwidths(p, t, BS_POS, n_tx))
        mis = np.array(mis)

        rows.append(dict(
            n_tx=n_tx, predictor=name,
            snr_db=float(np.mean([snr_db(s) for s in snrs])),
            se=float(np.mean([spectral_efficiency(s) for s in snrs])),
            misalign_bw=float(np.mean(mis)),
            outage=float(np.mean(mis > 0.5)),
        ))

    # Ceiling: beam computed from the true future channel.
    idx = twin.nearest_indices(truths_for(W_REF))
    orc = [snr(twin.channel_for_index(i), mrt_weights(twin.channel_for_index(i)),
               noise[n_tx]) for i in idx]
    rows.append(dict(n_tx=n_tx, predictor="oracle",
                     snr_db=float(np.mean([snr_db(s) for s in orc])),
                     se=float(np.mean([spectral_efficiency(s) for s in orc])),
                     misalign_bw=0.0, outage=0.0))
    print(f"Nt={n_tx} done")

In [ ]:
import collections
by_pred = collections.defaultdict(list)
for r in rows:
    by_pred[r["predictor"]].append(r)

header = "".join(f"{'Nt=' + str(n):>10}" for n in N_TX)
print(f"{'predictor':<12}{header}   (mean SNR, dB)")
for name, rs in by_pred.items():
    rs = sorted(rs, key=lambda r: r["n_tx"])
    print(f"{name:<12}" + "".join(f"{r['snr_db']:>10.2f}" for r in rs))

print(f"\n{'predictor':<12}{header}   (outage: >0.5 HPBW)")
for name, rs in by_pred.items():
    rs = sorted(rs, key=lambda r: r["n_tx"])
    print(f"{name:<12}" + "".join(f"{r['outage']:>10.1%}" for r in rs))

In [ ]:
COLOURS = {"lstm": BLUE, "minlstm": AQUA, "kalman": ORANGE,
           "ls_linear": "#8a8984", "reactive": "#c9c8c2", "oracle": INK}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))

for name, rs in by_pred.items():
    rs = sorted(rs, key=lambda r: r["n_tx"])
    reference = name in ("oracle", "reactive")
    style = dict(color=COLOURS.get(name, MUTED), marker="o",
                 ls="--" if reference else "-", lw=1.6 if reference else 2.0,
                 label=name)
    ax1.plot([r["n_tx"] for r in rs], [r["snr_db"] for r in rs], **style)
    if name != "oracle":
        ax2.plot([r["n_tx"] for r in rs], [r["misalign_bw"] for r in rs], **style)

ax1.set_xscale("log", base=2); ax1.set_xticks(N_TX); ax1.set_xticklabels(N_TX)
ax1.set_xlabel("TX antennas"); ax1.set_ylabel("mean SNR (dB)")
ax1.set_title("SNR vs array size"); ax1.legend(fontsize=8)

ax2.set_xscale("log", base=2); ax2.set_yscale("log")
ax2.set_xticks(N_TX); ax2.set_xticklabels(N_TX)
ax2.axhline(0.5, color=MUTED, ls=":", lw=1.2)
ax2.text(N_TX[0], 0.52, " half-power edge", fontsize=8, color=MUTED, va="bottom")
ax2.set_xlabel("TX antennas"); ax2.set_ylabel("misalignment (beamwidths)")
ax2.set_title("The metric only discriminates once beams are narrow")
plt.savefig(RESULTS / "nt_sweep.png"); plt.show()

## 6. Lead time

One slot of lookahead assumes the beam can be applied instantly. Real systems
need margin for beam training and processing, so the useful question is how many
slots ahead a predictor stays inside the beam. Error compounds over the horizon,
which is where architectures separate if they separate anywhere.

In [ ]:
HORIZONS = [1, 2, 5, 10]
lead = []
for h in HORIZONS:
    for name, (pred, W) in PREDICTORS.items():
        # Hold the beam for h slots: the t+1 prediction judged against truth at t+h.
        mis, start = [], 0
        for t in test_c:
            n = len(t["observed"]) - W
            for i in range(n - h + 1):
                mis.append(misalignment_beamwidths(
                    pred[start + i], t["true"][W + i + h - 1], BS_POS, 64))
            start += n
        mis = np.array(mis)
        lead.append(dict(horizon=h, predictor=name,
                         misalign_bw=float(np.mean(mis)),
                         outage=float(np.mean(mis > 0.5))))

head = "".join(f"{'+' + str(h):>10}" for h in HORIZONS)
print(f"{'predictor':<12}{head}   (outage at Nt=64)")
for name in PREDICTORS:
    vals = sorted((r for r in lead if r["predictor"] == name),
                  key=lambda r: r["horizon"])
    print(f"{name:<12}" + "".join(f"{r['outage']:>10.1%}" for r in vals))

In [ ]:
(RESULTS / "beamforming_eval.json").write_text(
    json.dumps(dict(nt_sweep=rows, lead_time=lead, bs_pos=BS_POS.tolist(),
                    leak_rate=hits / total), indent=2, default=float))
print(f"saved -> {RESULTS / 'beamforming_eval.json'}")

## 7. Reading the results

**Did fixing the leak change the story?** Compare the measured leak rate in §2
against the SNR table. If the old lookup returned the true index in most steps,
the published +0.05 to +0.11 dB gains were the oracle row of this table, not a
predictive result.

**Where does prediction start to matter?** Read the outage row across Nt. At
Nt=8 every predictor should look identical - 12.75 degrees of beamwidth absorbs
everything. The spread opens as beams narrow, and the array size where it opens
is the honest answer to "when is predictive beamforming worth it".

**Does the predictor ranking match the MAE ranking?** Not necessarily, and that
is worth a paragraph in the paper. MAE weights all errors equally; beamforming
only cares about the angular component, so lateral error at long range is nearly
free while the same error close to the base station is not. A model with worse
MAE can win on SNR.

**How much lead time is affordable?** Section 6 reports outage against horizon.
The largest horizon that keeps outage acceptable is the real system-level
result - that is the beam-training budget a scheduler can actually spend.